# Nash Equilibrium

For a 2-player, 2-action normal-form game, you can do it in two steps: check pure equilibria, then (if needed) solve for mixed.

Let player X choose rows $A_1, A_2$ and player Y choose columns $B_1, B_2$.  
Suppose you know the payoff matrices (returns) for each player:

$$
U^X =
\begin{pmatrix}
u^X_{11} & u^X_{12} \\
u^X_{21} & u^X_{22}
\end{pmatrix},
\quad
U^Y =
\begin{pmatrix}
u^Y_{11} & u^Y_{12} \\
u^Y_{21} & u^Y_{22}
\end{pmatrix}
$$

where $u^X_{ij}$ is X's payoff when X plays $A_i$ and Y plays $B_j$, and similarly for $u^Y_{ij}$.


In [2]:
import numpy as np

## Pure‑strategy Nash equilibria

A cell $(A_i, B_j)$ is a Nash equilibrium if:

- **X best response:** $u^X_{ij} \ge u^X_{kj}$ for all other rows $k$ (given column $j$).
- **Y best response:** $u^Y_{ij} \ge u^Y_{il}$ for all other columns $l$ (given row $i$).

So:

1. **Fix a column $j$** and look at X’s payoffs in that column; mark the row(s) with maximal payoff as X’s best response(s).
2. **Fix a row $i$** and look at Y’s payoffs in that row; mark the column(s) with maximal payoff as Y’s best response(s).
3. Any cell that is simultaneously a best response for both players is a pure Nash equilibrium.


In [3]:
def pure_nash_equilibria(Ux, Uy):
    """
    Find pure-strategy Nash equilibria in a 2x2 game.

    Ux[i, j] = payoff to player X when X plays action i and Y plays action j
    Uy[i, j] = payoff to player Y when X plays action i and Y plays action j
    """

    nash_eqs = []

    # Loop over all action pairs (i, j)
    for i in range(2):      # X's action
        for j in range(2):  # Y's action

            # Check if X is best-responding to Y's action j
            x_payoff = Ux[i, j]
            x_best_response = x_payoff >= Ux[:, j].max()

            # Check if Y is best-responding to X's action i
            y_payoff = Uy[i, j]
            y_best_response = y_payoff >= Uy[i, :].max()

            if x_best_response and y_best_response:
                nash_eqs.append((i, j))

    return nash_eqs

The output is a list of tuples $(i, j)$ where:
- $i$ is X's action index (0 or 1)
- $j$ is Y's action index (0 or 1)

In [5]:
Ux = np.array([[3, 1],
               [0, 2]])

Uy = np.array([[2, 0],
               [1, 3]])

print("Pure Nash equilibria:", pure_nash_equilibria(Ux, Uy))

Pure Nash equilibria: [(0, 0), (1, 1)]


- For column 1, best option for X is $(A_1, B_1)$, and for column 2, best option is $(A_2, B_2)$

- For row 1, best option for Y is $(A_1, B_1)$, and for row 2, best option is $(A_2, B_2)$

- Thus, the pure equilibrium, i.e., common options are $(A_1, B_1)$ and $(A_2, B_2)$.

In [6]:
Ux = np.array([[100, 0],
               [1, 1]])

Uy = np.array([[100, 0],
               [2, 1]])

print("Pure Nash equilibria:", pure_nash_equilibria(Ux, Uy))

Pure Nash equilibria: [(0, 0)]


- For column 1, best option for X is $(A_1, B_1)$, and for column 2, best option is $(A_2, B_2)$

- For row 1, best option for Y is $(A_1, B_1)$, and for row 2, best option is $(A_2, B_1)$

- Thus, the pure equilibrium, i.e., common options is $(A_1, B_1)$.

##  Mixed‑strategy Nash equilibrium
#### (if no pure, or you want all NE)

Let X mix: play $A_1$ with probability $p$, $A_2$ with probability $1-p$.
Let Y mix: play $B_1$ with probability $q$, $B_2$ with probability $1-q$.

At a mixed Nash equilibrium, each player must be indifferent between their two actions (otherwise they’d put probability 1 on the better one).

#### Y’s indifference (solve for $p$)

Y’s expected payoff from $B_1$:

$$
E_Y(B_1) = p \, u^Y_{11} + (1-p) \, u^Y_{21}
$$

Y’s expected payoff from $B_2$:

$$
E_Y(B_2) = p \, u^Y_{12} + (1-p) \, u^Y_{22}
$$

Indifference condition:

$$
p \, u^Y_{11} + (1-p) \, u^Y_{21}
=
p \, u^Y_{12} + (1-p) \, u^Y_{22}
$$

Solve for $p$:

$$
p^\ast
=
\frac{u^Y_{22} - u^Y_{21}}
{(u^Y_{11} - u^Y_{21}) - (u^Y_{12} - u^Y_{22})}
$$

(Algebra: collect terms in $p$ and divide.)

#### X's indifference (solve for $q$)

X's expected payoff from $A_1$:

$$
E_X(A_1) = q \, u^X_{11} + (1-q) \, u^X_{12}
$$

X's expected payoff from $A_2$:

$$
E_X(A_2) = q \, u^X_{21} + (1-q) \, u^X_{22}
$$

Indifference condition:

$$
q \, u^X_{11} + (1-q) \, u^X_{12}
=
q \, u^X_{21} + (1-q) \, u^X_{22}
$$

Solve for $q$:

$$
q^\ast
=
\frac{u^X_{22} - u^X_{12}}
{(u^X_{11} - u^X_{12}) - (u^X_{21} - u^X_{22})}
$$

### Validity of the mixed equilibrium

- If $p^\ast, q^\ast \in (0,1)$, you have an **interior mixed-strategy Nash equilibrium**.
- If either probability is $\le 0$ or $\ge 1$, then no interior mixed equilibrium exists; the Nash equilibria are pure (if any) and you fall back to the pure-strategy check.
- If denominators above are zero, you either have:
  - no interior mixed equilibrium, or
  - a continuum of equilibria along an edge (because of multiple best responses).


In [9]:
def mixed_nash_equilibrium(Ux, Uy):
    """
    Compute the mixed-strategy Nash equilibrium for a 2x2 game.

    Ux[i, j] = payoff to X when X plays i and Y plays j
    Uy[i, j] = payoff to Y when X plays i and Y plays j

    Returns (p, q) where:
        p = probability X plays action 0
        q = probability Y plays action 0
    If no valid mixed equilibrium exists, returns None.
    """

    # --- Solve for p (Y's indifference) ---
    # Y indifferent between B1 and B2:
    # p * Uy[0,0] + (1-p) * Uy[1,0] = p * Uy[0,1] + (1-p) * Uy[1,1]

    a = Uy[0,0] - Uy[1,0]
    b = Uy[0,1] - Uy[1,1]

    denom_p = a - b
    if denom_p == 0:
        p = None
    else:
        p = (Uy[1,1] - Uy[1,0]) / denom_p

    # --- Solve for q (X's indifference) ---
    # X indifferent between A1 and A2:
    # q * Ux[0,0] + (1-q) * Ux[0,1] = q * Ux[1,0] + (1-q) * Ux[1,1]

    c = Ux[0,0] - Ux[0,1]
    d = Ux[1,0] - Ux[1,1]

    denom_q = c - d
    if denom_q == 0:
        q = None
    else:
        q = (Ux[1,1] - Ux[0,1]) / denom_q

    # Check validity: probabilities must be in (0,1)
    if p is None or q is None or not (0 < p < 1) or not (0 < q < 1):
        return None

    return p, q

In [10]:
Ux = np.array([[3, 1],
               [0, 2]])

Uy = np.array([[2, 0],
               [1, 3]])

print("Mixed Nash equilibrium:", mixed_nash_equilibrium(Ux, Uy))

Mixed Nash equilibrium: (np.float64(0.5), np.float64(0.25))
